
## 一、核心业务背景
- **案例场景**：网约车5大营销渠道（A、B、C、D、E）的投放效果评估。
- **业务目标**：在有限预算下，选择**高转化、低成本、高性价比**的渠道进行追加投入。
- **关键指标体系**：
  - **过程指标**：注册数、首呼数、首单数（易造假，需警惕）。
  - **结果指标**：最终转化率、每首单成本（核心决策标准）。

---

## 二、13.1 漏斗分析法
### 1. 核心概念
- **定义**：将业务流程（投放→注册→首呼→首单）可视化，展示每个环节的流失率，定位转化瓶颈。
- **三大优势**：
  1.  **可视化**：直观看到用户流失环节。
  2.  **定量评估**：对比不同渠道的转化策略效果。
  3.  **预测优化**：根据转化率预测未来趋势，制定优化策略。

### 2. 关键转化公式
| 指标 | 公式 | 含义 |
|------|------|------|
| **首呼转化率** | `(首呼数 / 注册数) × 100%` | 注册用户中愿意下单呼叫的比例 |
| **最终转化率** | `(首单数 / 注册数) × 100%` | 注册用户中完成首单的最终比例（**核心结果指标**） |
| **每首单成本** | `总投入成本 / 首单数` | 获客成本（越低越好） |

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('广告营销渠道数据.csv')
df.head(2)

,渠道,注册数,首呼数,首单数,已投入成本
0,A,12668,9499,8358,259201
1,B,22762,16386,13599,319915


In [23]:
df['首呼转化率'] = round((df['首呼数']/df['注册数'])*100, 2)
df['首单转化率'] = round(df['首单数']/df['首呼数']*100,2)
df['最终转化率'] = round(df['首单数']/df['注册数']*100,2)
df

,渠道,注册数,首呼数,首单数,已投入成本,首呼转化率,首单转化率,最终转化率
0,A,12668,9499,8358,259201,74.98,87.99,65.98
1,B,22762,16386,13599,319915,71.99,82.99,59.74
2,C,6872,4395,3734,116847,63.96,84.96,54.34
3,D,55436,37694,34678,651413,68.00,92.00,62.56
4,E,34189,25297,22007,133628,73.99,86.99,64.37


In [24]:
df['每注册成本'] = round(df['已投入成本']/df['注册数'],2)
df['每首呼成本'] = round(df['已投入成本']/df['首呼数'],2)
df['每首单成本'] = round(df['已投入成本']/df['首单数'],2)
df[['渠道','首呼转化率','首单转化率','最终转化率','每注册成本','每首呼成本','每首单成本']]

,渠道,首呼转化率,首单转化率,最终转化率,每注册成本,每首呼成本,每首单成本
0,A,74.98,87.99,65.98,20.46,27.29,31.01
1,B,71.99,82.99,59.74,14.05,19.52,23.52
2,C,63.96,84.96,54.34,17.00,26.59,31.29
3,D,68.00,92.00,62.56,11.75,17.28,18.78
4,E,73.99,86.99,64.37,3.91,5.28,6.07


In [25]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 准备数据（假设 df 已包含各渠道数据）
channels = df['渠道'].tolist()
stages = ['注册数', '首呼数', '首单数']

# 创建子图，2行3列（5个渠道占满）
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=channels,
                    specs=[[{'type': 'funnel'}, {'type': 'funnel'}, {'type': 'funnel'}],
                           [{'type': 'funnel'}, {'type': 'funnel'}, {'type': 'funnel'}]],
                    horizontal_spacing=0.08, vertical_spacing=0.15)

# 为每个渠道添加漏斗图
for i, ch in enumerate(channels):
    row = i // 3 + 1
    col = i % 3 + 1
    values = [df.loc[df['渠道'] == ch, s].values[0] for s in stages]

    fig.add_trace(go.Funnel(
        name=ch,
        y=stages,
        x=values,
        textinfo="value+percent previous",
        textposition="inside",
        marker=dict(color="skyblue"),
        connector=dict(line=dict(color="royalblue", width=2))
    ), row=row, col=col)

fig.update_layout(height=800, showlegend=False, title_text="各渠道转化漏斗对比")
fig.show()

### 4. 业务洞察：漏斗分析结果
| 渠道 | 核心表现 | 评价 |
|------|----------|------|
| **A** | 最终转化率最高(65.98%)，但成本极高(31.01) | 质量高但昂贵，性价比一般 |
| **E** | 成本最低(6.07)，转化率也不错(64.37%) | **性价比之王**，首选 |
| **D** | 成本低(18.78)，转化率稳健(62.56) | **潜在黑马**，次选 |
| **C** | 转化率最低(54.34)，成本高(31.29) | 效率最低，需调整 |

---

---

## 三、13.2 整体结构分析法
### 1. 为什么引入整体平均？
- **过程指标造假**：注册数、首呼数可通过机器人刷量，不可信。
- **成本造假**：每注册成本可通过低价引流刷高，不可信。
- **最终结论**：必须结合**最终转化率**和**每首单成本**，并引入**整体平均水平**作为基准。

### 2. 核心分析逻辑
1.  **计算整体平均**：作为行业/全局参考基准。
2.  **差值分析**：对比各渠道与整体平均的差距（正负值）。
3.  **性价比决策**：
    - 高转化 + 低成本 = 优质渠道（优先投入）
    - 高转化 + 高成本 = 看增量产出（权衡投入）
    - 低转化 + 低成本 = 优化空间大（调整策略）
    - 低转化 + 高成本 = 劣质渠道（减少投入）



### 3. Python代码实现（整体对标）

In [26]:
# 构建一个新的DataFrame，将渠道列的值改为整体平均四个字，其余列名不变
total = pd.DataFrame(columns = df.columns)
total['渠道'] = ['整体平均']

for col in df.drop(columns=['渠道']).columns.tolist():
    total[col] = np.mean(df[col])

df = pd.concat([df,total],ignore_index=True)
df[['渠道','最终转化率','每首单成本']]

,渠道,最终转化率,每首单成本
0,A,65.980,31.010
1,B,59.740,23.520
2,C,54.340,31.290
3,D,62.560,18.780
4,E,64.370,6.070
5,整体平均,61.398,22.134


计算与整体的差值

In [27]:
# 1. 先计算差值（保持数值类型）
df['最终转化率与整体'] = df.iloc[:5]['最终转化率'] - df.iloc[-1,:]['最终转化率']
df['每首单成本与整体'] = df.iloc[:5]['每首单成本'] - df.iloc[-1,:]['每首单成本']

# 2. 格式化时跳过 NaN
df['最终转化率'] = df['最终转化率'].apply(lambda x: f"{round(x)}%" if pd.notna(x) else "")
df['最终转化率与整体'] = df['最终转化率与整体'].apply(lambda x: f"{round(x)}%" if pd.notna(x) else "")

# 3. 展示结果（round 对字符串列无意义，直接取列即可）
result_df = df[['渠道', '最终转化率', '每首单成本', '最终转化率与整体', '每首单成本与整体']]
result_df

,渠道,最终转化率,每首单成本,最终转化率与整体,每首单成本与整体
0,A,66%,31.010,5%,8.876
1,B,60%,23.520,-2%,1.386
2,C,54%,31.290,-7%,9.156
3,D,63%,18.780,1%,-3.354
4,E,64%,6.070,3%,-16.064
5,整体平均,61%,22.134,,NaN


### 4. 最终决策结论
| 渠道 | 与整体对比 | 最终判定 | 策略建议 |
|------|------------|----------|----------|
| **A** | 转化高5%，成本高9.0 | 高投入高回报 | 可追加投入，但需控制成本 |
| **E** | **转化高4%，成本低16.0** | **绝对最优** | **加大投入**，性价比最高 |
| **D** | 转化高1%，成本低3.0 | 稳健优质 | 追加投入，巩固优势 |
| **B** | 转化低2%，成本低1.0 | 平庸 | 维持现状或小幅优化 |
| **C** | 转化低7%，成本高9.0 | 低效淘汰 | 减少投入，重点调整优化 |




## 四、13.3 渠道分析核心总结
### 1. 三大分析方法
1.  **漏斗分析法**：看过程，找流失点（评估转化策略）。
2.  **成本分析法**：看投入，算回报（评估ROI）。
3.  **整体结构分析法**：看基准，做决策（引入整体平均，避免造假与片面）。

### 2. 避坑指南
- **警惕过程指标**：注册数、点击率、首呼数极易造假，不能作为最终投放标准。
- **聚焦结果指标**：必须看**最终转化率**（质量）和**每首单成本**（效率）。
- **结合整体平均**：没有对比就没有伤害，引入整体水平能发现被忽略的优质渠道（如本章的渠道E）。

### 3. 最终营销决策
- **首选渠道**：**E**（转化率高、成本极低，性价比爆表）。
- **备选渠道**：**D**（成本低、转化稳健）。
- **慎用渠道**：**A**（虽然转化最高，但成本太高，ROI不如E）。
- **放弃渠道**：**C**（转化低、成本高，效率最低）。

---



### 完整分析流程速记
1.  **建漏斗**：计算各环节转化率，可视化转化路径。
2.  **算成本**：计算每注册、每首单成本，评估投放性价比。
3.  **去伪存真**：过滤造假风险高的过程指标。
4.  **找标杆**：引入整体平均水平，作为决策基准。
5.  **下结论**：综合最终转化率与成本，输出最优渠道组合。